In [53]:
import numpy as np
import tensorflow as tf
import h5py
import os
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, BatchNormalization
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pickle
import json
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns


In [36]:
pip install regex

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [37]:
pip install logparser3 --no-deps

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [38]:
pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [39]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install scikit-learn

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [41]:
from logparser.Drain import LogParser
print("Drain working")
import pandas as pd
import re
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

Drain working


In [43]:
log_path = "HDFS.log"
label_path = "anomaly_label.csv"
with open(log_path, "r") as f:
    for _ in range(5):
        print(f.readline())

081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010

081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906

081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010

081109 203519 145 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010

081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687919862906 terminating



In [44]:
log_format = '<Date> <Time> <Pid> <Level> <Component>: <Content>'
input_dir = "."          # current folder
output_dir = "."  # create output folder here
parser = LogParser(
    log_format=log_format,
    indir=input_dir,
    outdir=output_dir,
    depth=4,
    st=0.5
)

In [45]:
parser.parse('HDFS.log')

Parsing file: ./HDFS.log
Total lines:  11175629
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.0% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.1% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.2% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log lines.
Processed 0.3% of log li

In [46]:
df = pd.read_csv('HDFS.log_structured.csv')
def extract_block_id(text):
    match = re.search(r'(blk_-?\d+)', str(text))
    return match.group(1) if match else None
df['BlockId'] = df['Content'].apply(extract_block_id)
df[['Content', 'BlockId']].head()

,Content,BlockId
0,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
1,BLOCK* NameSystem.allocateBlock: /mnt/hadoop/m...,blk_-1608999687919862906
2,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
3,Receiving block blk_-1608999687919862906 src: ...,blk_-1608999687919862906
4,PacketResponder 1 for block blk_-1608999687919...,blk_-1608999687919862906


In [47]:
df = df[df['BlockId'].notnull()]
df = df.sort_values(by='LineId')
grouped = df.groupby('BlockId')['EventId'].apply(list)
data = pd.DataFrame({
    'BlockId': grouped.index,
    'Sequence': grouped.values
})
labels = pd.read_csv('anomaly_label.csv')
labels['Label'] = labels['Label'].map({
    'Normal': 0,
    'Anomaly': 1
})
data = data.merge(labels, on='BlockId')
data = data[['BlockId', 'Sequence', 'Label']]

In [48]:
mapping_path = 'event2idx.pkl'
if not os.path.exists(mapping_path):
    mapping_path = 'research/event2idx.pkl'

with open(mapping_path, 'rb') as f:
    event2idx = pickle.load(f)

print('Loaded event2idx from', mapping_path)
data['Sequence'] = data['Sequence'].apply(
    lambda seq: [event2idx[event] for event in seq]
)

Loaded event2idx from research/event2idx.pkl


In [ ]:
#THE SPECIFIC CODE FOR SPLITTING THE DATA INTO TRAIN, VAL, AND TEST SETS IS AS FOLLOWS:
'''
from sklearn.model_selection import train_test_split
# First split: train vs temp
X_train, X_temp, y_train, y_temp = train_test_split(
    data['Sequence'],
    data['Label'],
    test_size=0.9,
    random_state=42
)
# Second split: val vs test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.,
    random_state=42
)
max_len = 50
X_train = pad_sequences(X_train, maxlen=max_len, padding='post')
X_val   = pad_sequences(X_val,   maxlen=max_len, padding='post')
X_test  = pad_sequences(X_test,  maxlen=max_len, padding='post')
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)
'''

#OR IF YOU WANT THE ENTIRE DTATASET
X_test = data['Sequence']
y_test = data['Label']

max_len = 50
X_test = pad_sequences(X_test, maxlen=max_len, padding='post')

print(X_test.shape)
print(y_test.shape)

(575061, 50)
(575061,)


In [66]:
model = load_model("models/lstm_model(v2).keras", compile=False)
print("Loaded model from models/lstm_model(v2).keras")


Loaded model from models/lstm_model(v2).keras


In [71]:
print("from root directory")

y_test_pred_prob = model.predict(X_test, batch_size=256)
y_test_pred = (y_test_pred_prob > 0.5).astype(int)
print("=== TEST CONFUSION MATRIX ===")
cm = confusion_matrix(y_test, y_test_pred)
print(cm)

print("\n=== TEST CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_test_pred))

print("Test F1 Score:", f1_score(y_test, y_test_pred))

from root directory
2247/2247 ━━━━━━━━━━━━━━━━━━━━ 78s 35ms/step
=== TEST CONFUSION MATRIX ===
[[557829    394]
 [     8  16830]]

=== TEST CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    558223
           1       0.98      1.00      0.99     16838

    accuracy                           1.00    575061
   macro avg       0.99      1.00      0.99    575061
weighted avg       1.00      1.00      1.00    575061

Test F1 Score: 0.9881979918971288
